## Futásidő-összevetés: rprev referencia (naiv K-szoros) vs. kiterjesztett több indexdátumos implementáció

Ez a notebook a szakdolgozat 5. és 7. fejezetében rögzített több indexdátumos kiterjesztéshez kapcsolódó **futásidő-mérési protokollt** valósítja meg. A mérés célja annak kvantifikálása, hogy azonos bemeneti adaton és azonos szimulációs beállítások mellett milyen számítási igénye van  
(i) a referencia-megoldásnak, ahol a $\{t_k\}_{k=1}^K$ rácsot **$K$ külön `prevalence()` függvényhívással** állítjuk elő, illetve  
(ii) a kiterjesztett rprev-implementációnak, ahol az `index` argumentum több időpont fogadására is képes, és a becslés **egyetlen eljárásfuttatásban** állítja elő a teljes prevalenciavektort és annak összefoglalóját.

### Kapcsolódás a dolgozathoz

A kiterjesztés matematikai tartalma röviden: adott $t_1 < \cdots < t_K$ rácson, az $r$-edik Monte Carlo futásban a futáson belül előállított incidens esetek $(D_{i\ell}^{(r)}, \mathbf{x}_{i\ell}^{(r)})$ és a futáson belül rögzített túlélési függvény $S^{(r)}(\cdot \mid \mathbf{x})$ alapján minden $t_k$ időpontra képezzük a prevalens státuszt, majd időpontonként összegezve kapjuk a $P^{(r)}(t_k)$ prevalenciakomponenseket és a futások feletti összefoglaló statisztikákat.

Csomagszinten a notebook azt a működést tekinti „kiterjesztett” megvalósításnak, ahol a `prevalence()` az `index` bemenetet rendezett $\texttt{index\_dates} = (t_1,\ldots,t_K)$ vektorrá normalizálja, a belső szimulációs réteg futásonként az `index_dates` rácson szolgáltat státuszinformációt, amelyből az összegző réteg időpontonként képez prevalencia-összefoglalót. A kimeneti objektum tartalmazza az `index_dates` vektort és az időpontindexelt eredményeket.

### Mérési egységek és protokoll

A dolgozat terminológiáját követve:

- **Alapkiértékelés:** a `prevalence()` egy futtatása rögzített $t$ mellett.
- **Eljárásfuttatás:** a $\{t_k\}_{k=1}^K$ időpontrács teljes kimenetének előállítása.

A naív, referencia-megoldásban egy eljárásfuttatás **$K$ darab alapkiértékelésből** áll (külön-külön időpontra), míg a kiterjesztett, natív módon több indexpontot támogató megoldásban ugyanez **egyetlen alapkiértékelésként** valósul meg.

A notebook minden paraméterkombinációra ismételt futásidő-mérést végez (`reps`), és az összidőt `system.time(...)[["elapsed"]]` alapján rögzíti. A riportolt érték minden beállításpontra azok egyes ismételt futásaiból származó eredményeinek átlaga.

A mérési protokoll kizárólag a futásidő kvantifikálására szolgál. A prevalenciabecslés pontosságának vizsgálata ettől elkülönítve, egy másik notebookban történik.

### Bemeneti adat és indexdátumok

A méréshez az rprev csomag `prevsim` példaadata szolgál alapul, melyből visszatevéses mintavétellel áll elő az $N$ soros, regiszterjellegű bemeneti tábla (a mérésben az $N$ a rekordszámot jelöli). Az indexdátumok egy rögzített kezdődátumtól (`index_start`) fix naplépéssel (`index_step_days`) generált, rendezett $K$ hosszú sorozatot alkotnak.

A notebookban az indexdátumok generálása determinisztikus: adott $K$, `index_start` és `index_step_days` mellett az `index_dates` teljesen reprodukálható, és a különböző mérési pontok között kizárólag a konfigurációs paraméterek változnak.

A bemeneti táblát a futásidő skálázódásának vizsgálata motiválja, ezért az adat-előállítás során nem cél egy valós regiszter teljes, esetszintű konzisztenciájának biztosítása. A visszatevéses mintavétel következtében előfordulhatnak ismétlődő rekordok, ezért a kapott állomány nem tekintendő tiszta regiszternek a szokásos adatminőségi kritériumok szerint. A mérés érvényességét viszont ez nem érinti, mivel a vizsgálat tárgya kizárólag a számítási terhelés és annak $N$ és $K$ szerinti alakulása.

### Szimulációs és bootstrap beállítások

A mérések rácsa a következő paraméterek mentén szervezett:

- $N$: a bemeneti tábla sorainak száma (adatméret, rekordszám),
- $K$: az indexdátumok száma,
- `N_boot`: a `prevalence()` hívásban használt bootstrap ismétlések száma,
- `reps`: az időmérés ismétlésszáma paraméterpontonként.

A fenti rács mellett a hívásokban több beállítás rögzített, és a futtatások között nem változik: az indexdátum-generálás paraméterei, valamint a túlélési eloszlás választása.

### Kimenet

A futás végén egy eredménytábla készül, amely minden paraméterpontra tartalmazza:

- `N`, `K`, `N_boot`,
- `reps`,
- `elapsed_sec_mean`: az eljárásfuttatás átlagos futásideje másodpercben.

---


#### 1. Közös paraméter konfiguráció
- A futásidő-mérés paraméterrácsának és a rögzített modellspecifikációs beállításoknak a központi definiálása a reprodukálhatóság érdekében.
- A referencia és a kiterjesztett megoldása futtatása ugyanebből a paraméterkészletből dolgozik, így az összevetés konzisztens.

In [3]:
COMMON_CFG <- list(
  # Méretek / rácsparaméterek
  N_values              = c(1e2, 1e3),        # bemeneti tábla sorszámai (rekordszámok)
  K_values              = c(1, 5),            # indexdátumok száma (K időpont)
  N_boot_values         = c(50, 100),         # bootstrap ismétlések száma
  reps                  = 3L,                 # ismétlések száma paraméterpontonként (átlagolt idő)
  num_years_to_estimate = c(15),              # becslési horizont (év)
  index_start           = as.Date("2013-01-30"), # első indexdátum
  index_step_days       = 30L,                # indexdátumok lépésköze napokban
  dist                  = "weibull",          # túlélési eloszlás típusa
  population_size       = 1e6                 # populációméret a prevalenciához
)

# COMMON_CFG <- list(
#   # Méretek / rácsparaméterek
#   N_values              = c(1e2, 1e3),        # bemeneti tábla sorszámai (rekordszámok)
#   K_values              = c(1, 10, 20, 30),   # indexdátumok száma (K időpont)
#   N_boot_values         = c(100, 500, 1000),  # bootstrap ismétlések száma
#   reps                  = 3L,                 # ismétlések száma paraméterpontonként (átlagolt idő)
#   num_years_to_estimate = c(15),              # becslési horizont (év)
#   index_start           = as.Date("2013-01-30"), # első indexdátum
#   index_step_days       = 30L,                # indexdátumok lépésköze napokban
#   dist                  = "weibull",          # túlélési eloszlás típusa
#   population_size       = 1e6                 # populációméret a prevalenciához
# )

#### 2. Környezet előkészítése és skálázott bemenetek képzése
- A szükséges csomagok betöltése után a `prevsim` példaadat kerül beolvasásra.
- A `prevsim` táblából, a konfigurációban megadott `N_values` értékekre visszatevéses mintavétellel több, kontrollált méretű bemeneti tábla készül, melyek egységes alapot biztosítanak a futásidőmérésekhez.

In [4]:
# Csomagok betöltése
suppressPackageStartupMessages({
  library(rprev)
  library(survival)
})

# Példaadat betöltése (kiinduló tábla)
data(prevsim, package = "rprev")
base_df <- as.data.frame(prevsim)

# Skálázott bemenetek előállítása: minden N-hez egy N soros minta (visszatevéses)
scaled_data <- setNames(
  lapply(COMMON_CFG$N_values, function(N) {
    set.seed(10000L + as.integer(N))  # reprodukálhatóság N-enként
    base_df[sample.int(nrow(base_df), size = as.integer(N), replace = TRUE), , drop = FALSE]
  }),
  paste0("N", COMMON_CFG$N_values)    # listanevek: "N100", "N1000", ...
)


Warning message:
"package 'rprev' was built under R version 4.4.3"


#### 3. Referencia-megoldás futásidő-mérése: K külön `prevalence()` hívás
- A segédfüggvények előállítják az $N$ soros bemenetet és a $K$ darab indexdátumot, majd a referencia-megoldás $K$ külön hívással elvégzi a számítást.
- Az időmérés `system.time()` alapján történik, és paraméterpontonként az `reps` ismétlésből származó futásidők kerülnek átlagolásra .

In [5]:
# ---- Bemenet- és indexdátum-generálás ----
make_registry <- function(N, seed = 1L) {
  set.seed(as.integer(seed))  # reprodukálhatóság
  base_df[sample.int(nrow(base_df), size = as.integer(N), replace = TRUE), , drop = FALSE]
}

make_index_dates <- function(K, start_date, step_days) {
  seq.Date(from = start_date, by = as.integer(step_days), length.out = as.integer(K))
}

# ---- Referencia-megoldás: egy prevalence() hívás egy indexdátumra ----
# index_date_chr: "YYYY-MM-DD"
run_rprev_once <- function(dat, index_date_chr, N_boot) {
  rprev::prevalence(
    index = index_date_chr,
    num_years_to_estimate = COMMON_CFG$num_years_to_estimate,
    data = dat,
    inc_formula  = entrydate ~ sex,
    surv_formula = Surv(time, status) ~ age + sex,
    dist = COMMON_CFG$dist,
    population_size = COMMON_CFG$population_size,
    death_column = "eventdate",
    N_boot = N_boot
  )
}

# ---- Referencia-megoldás időmérése: K külön hívás összideje ----
measure_one_setting <- function(N, K, N_boot, seed = 1L) {
  dat <- scaled_data[[paste0("N", as.integer(N))]]  # előre legenerált N-méretű bemenet

  idx_dates <- make_index_dates(K, COMMON_CFG$index_start, COMMON_CFG$index_step_days)
  idx_dates_chr <- format(idx_dates, "%Y-%m-%d")   # stabil dátumátadás (Date->chr)

  if (length(idx_dates_chr) != K || anyNA(idx_dates_chr)) {
    stop("Index dátum generálás hibás: idx_dates_chr NA-t tartalmaz vagy rossz hossz.")
  }

  elapsed <- numeric(COMMON_CFG$reps)
  for (r in seq_len(COMMON_CFG$reps)) {
    gc()  # ismétlések közti memóriahatás csökkentése
    t <- system.time({
      for (d_chr in idx_dates_chr) {
        invisible(run_rprev_once(dat, d_chr, N_boot))
      }
    })
    elapsed[r] <- unname(t[["elapsed"]])
  }

  data.frame(
    N = N,
    K = K,
    N_boot = N_boot,
    reps = COMMON_CFG$reps,
    elapsed_sec_mean = mean(elapsed)
  )
}

# ---- Paraméterrács futtatása és eredménytábla előállítása ----
results <- do.call(
  rbind,
  lapply(COMMON_CFG$N_values, function(N) {
    do.call(rbind, lapply(COMMON_CFG$K_values, function(K) {
      do.call(rbind, lapply(COMMON_CFG$N_boot_values, function(N_boot) {
        message(sprintf("Run: N=%g, K=%d, N_boot=%d", N, K, N_boot))
        measure_one_setting(
          N = N,
          K = K,
          N_boot = N_boot,
          seed = 1000 + as.integer(N) + as.integer(K) + as.integer(N_boot)
        )
      }))
    }))
  })
)

results <- results[order(results$N, results$K, results$N_boot), ]
within(results, { elapsed_sec_mean <- round(elapsed_sec_mean, 2) })

# write.csv(results, "rprev_runtime_pilot.csv", row.names = FALSE)

Run: N=100, K=1, N_boot=50

Run: N=100, K=1, N_boot=100

Run: N=100, K=5, N_boot=50

Run: N=100, K=5, N_boot=100

Run: N=1000, K=1, N_boot=50

Run: N=1000, K=1, N_boot=100

Run: N=1000, K=5, N_boot=50

Run: N=1000, K=5, N_boot=100



,N,K,N_boot,reps,elapsed_sec_mean
,<dbl>,<dbl>,<dbl>,<int>,<dbl>
1,100,1,50,3,0.47
2,100,1,100,3,0.54
3,100,5,50,3,1.51
4,100,5,100,3,3.44
5,1000,1,50,3,0.62
6,1000,1,100,3,0.96
7,1000,5,50,3,2.65
8,1000,5,100,3,5.35


#### 4. Kiterjesztett rprev csomag betöltése lokális forrásból és git-meta rögzítése
- A csomag lokális forrásból kerül betöltésre, így a futtatás ténylegesen a vizsgált, módosított implementációval történik.
- A futtatott verzió azonosíthatóságát a git ág és commit azonosító kiírása biztosítja, ami az eredmények dokumentálásához szükséges.

In [6]:
# ---- Lokális rprev betöltése a repozitóriumból ----

# Projekt gyökérkönyvtárának meghatározása (git repo root)
project_root <- trimws(system2("git", c("rev-parse", "--show-toplevel"), stdout = TRUE))
if (length(project_root) == 0 || !dir.exists(project_root)) {
  stop("Could not resolve project root from git.")
}

# Telepített rprev leválasztása, majd lokális forrásból betöltés
if ("package:rprev" %in% search()) detach("package:rprev", unload = TRUE, character.only = TRUE)
if ("rprev" %in% loadedNamespaces()) unloadNamespace("rprev")
if (!requireNamespace("devtools", quietly = TRUE)) stop("devtools package is not available.")

suppressPackageStartupMessages(devtools::load_all(project_root, quiet = TRUE))

# Betöltött csomag elérési útjának megjelenítése
cat(
  "Loaded rprev from: ",
  normalizePath(getNamespaceInfo("rprev", "path"), winslash = "/"),
  "\n",
  sep = ""
)

Warning message:
"package 'testthat' was built under R version 4.4.3"


Loaded rprev from: C:/Users/600972868/OneDrive - BT Plc/Desktop/Sajat Mappa/00 - BGE/Alkalmazott Matematika/Szakdolgozat/repo/rprev-ext


In [7]:
# ---- Git meta rögzítése: futtatott kódverzió azonosítása ----
git_cmd <- function(args) {
  out <- suppressWarnings(
    tryCatch(system2("git", args, stdout = TRUE, stderr = TRUE), error = function(e) character(0))
  )
  status <- attr(out, "status")
  if (!is.null(status) || length(out) == 0) return(NA_character_)
  trimws(out[[1]])
}

old_wd <- getwd()                         # munkakönyvtár mentése
on.exit(setwd(old_wd), add = TRUE)        # visszaállítás a cella végén
setwd(project_root)                       # git parancsok futtatása a repo gyökeréből

branch <- git_cmd(c("rev-parse", "--abbrev-ref", "HEAD"))
commit <- git_cmd(c("rev-parse", "HEAD"))

cat("Git branch: ", branch, "\n", sep = "")
cat("Git commit: ", commit, "\n", sep = "")

Git branch: notebooks/runtime-benchmarks
Git commit: b07509be3ad072130348e92af7a3f5a905580159


#### 5. Kiterjesztett megoldás futásidő-mérése: natív több indexdátumos `prevalence()` hívás
- Az eljárás egyetlen `prevalence()` hívásban kapja meg a $K$ darab indexdátumot, és ugyanebben a futtatásban állítja elő a teljes kimenetet.
- Az időmérés azonos módon, `system.time()` alapján történik, és `reps` ismétlés átlaga kerül riportolásra.

In [8]:
# ---- Kiterjesztett megoldás: egy prevalence() hívás több indexdátumra ----
run_rprev_ext_once <- function(dat, index_dates_chr, N_boot) {
  rprev::prevalence(
    index = index_dates_chr,
    num_years_to_estimate = COMMON_CFG$num_years_to_estimate,
    data = dat,
    inc_formula  = entrydate ~ sex,
    surv_formula = Surv(time, status) ~ age + sex,
    dist = COMMON_CFG$dist,
    population_size = COMMON_CFG$population_size,
    death_column = "eventdate",
    N_boot = N_boot
  )
}

# ---- Kiterjesztett megoldás időmérése: egy hívás ideje ----
measure_one_setting_ext <- function(N, K, N_boot, seed = 1L) {
  dat <- scaled_data[[paste0("N", as.integer(N))]]  # előre legenerált N-méretű bemenet

  idx_dates <- make_index_dates(K, COMMON_CFG$index_start, COMMON_CFG$index_step_days)
  idx_dates_chr <- format(idx_dates, "%Y-%m-%d")   # stabil dátumátadás (Date->chr)

  if (length(idx_dates_chr) != K || anyNA(idx_dates_chr)) {
    stop("Index dátum generálás hibás: idx_dates_chr NA-t tartalmaz vagy rossz hossz.")
  }

  # Időmérés: reps ismétlésből az idő átlaga, majd eredménysor visszaadása
  elapsed <- numeric(COMMON_CFG$reps)
  for (r in seq_len(COMMON_CFG$reps)) {
    gc()
    t <- system.time({
      invisible(run_rprev_ext_once(dat, idx_dates_chr, N_boot))
    })
    elapsed[r] <- unname(t[["elapsed"]])
  }

  data.frame(
    N = N,
    K = K,
    N_boot = N_boot,
    reps = COMMON_CFG$reps,
    elapsed_sec_mean = mean(elapsed)
  )
}

# ---- Paraméterrács futtatása és eredménytábla előállítása ----
results_ext <- do.call(
  rbind,
  lapply(COMMON_CFG$N_values, function(N) {
    do.call(rbind, lapply(COMMON_CFG$K_values, function(K) {
      do.call(rbind, lapply(COMMON_CFG$N_boot_values, function(N_boot) {
        message(sprintf("Ext run: N=%g, K=%d, N_boot=%d", N, K, N_boot))
        measure_one_setting_ext(
          N = N,
          K = K,
          N_boot = N_boot,
          seed = 2000 + as.integer(N) + as.integer(K) + as.integer(N_boot)
        )
      }))
    }))
  })
)

results_ext <- results_ext[order(results_ext$N, results_ext$K, results_ext$N_boot), ]
within(results_ext, { elapsed_sec_mean <- round(elapsed_sec_mean, 2) })

Ext run: N=100, K=1, N_boot=50

Ext run: N=100, K=1, N_boot=100

Ext run: N=100, K=5, N_boot=50

Ext run: N=100, K=5, N_boot=100

Ext run: N=1000, K=1, N_boot=50

Ext run: N=1000, K=1, N_boot=100

Ext run: N=1000, K=5, N_boot=50

Ext run: N=1000, K=5, N_boot=100



,N,K,N_boot,reps,elapsed_sec_mean
,<dbl>,<dbl>,<dbl>,<int>,<dbl>
1,100,1,50,3,0.39
2,100,1,100,3,0.62
3,100,5,50,3,0.58
4,100,5,100,3,1.17
5,1000,1,50,3,0.53
6,1000,1,100,3,1.04
7,1000,5,50,3,0.99
8,1000,5,100,3,1.97


#### 6. Összesítés: referencia- és kiterjesztett futásidők összevetése
- A két eredménytábla paraméterpontonként összeillesztésre kerül, így közvetlenül összehasonlítható a referencia- és a kiterjesztett megoldás átlagos futásideje.
- A különbség abszolút (`delta_sec`) és relatív (`delta_pct`) formában is kiszámításra kerül, ami a gyorsulás mértékét számszerűsíti.

In [9]:
# ---- Összesítés: referencia- és több indexdátumos futásidők összevezetése és különbségek számítása ----
summary_runtime <- merge(
  results[, c("N", "K", "N_boot", "reps", "elapsed_sec_mean")],
  results_ext[, c("N", "K", "N_boot", "reps", "elapsed_sec_mean")],
  by = c("N", "K", "N_boot", "reps"),
  suffixes = c("_ref", "_multi")
)

summary_runtime <- summary_runtime[order(summary_runtime$N, summary_runtime$K, summary_runtime$N_boot), ]

# Abszolút és relatív eltérés (több indexdátum - referencia)
summary_runtime$delta_sec <- summary_runtime$elapsed_sec_mean_multi - summary_runtime$elapsed_sec_mean_ref
summary_runtime$delta_pct <- 100 * summary_runtime$delta_sec / summary_runtime$elapsed_sec_mean_ref

# Megjelenítés: kerekítés a táblázatos riporthoz
summary_runtime_disp <- within(summary_runtime, {
  elapsed_sec_mean_ref <- round(elapsed_sec_mean_ref, 2)
  elapsed_sec_mean_multi <- round(elapsed_sec_mean_multi, 2)
  delta_sec <- round(delta_sec, 2)
  delta_pct <- round(delta_pct, 2)
})

summary_runtime_disp

,N,K,N_boot,reps,elapsed_sec_mean_ref,elapsed_sec_mean_multi,delta_sec,delta_pct
,<dbl>,<dbl>,<dbl>,<int>,<dbl>,<dbl>,<dbl>,<dbl>
2,100,1,50,3,0.47,0.39,-0.08,-17.61
1,100,1,100,3,0.54,0.62,0.08,15.53
4,100,5,50,3,1.51,0.58,-0.94,-61.89
3,100,5,100,3,3.44,1.17,-2.27,-65.92
6,1000,1,50,3,0.62,0.53,-0.09,-14.52
5,1000,1,100,3,0.96,1.04,0.08,8.30
8,1000,5,50,3,2.65,0.99,-1.66,-62.56
7,1000,5,100,3,5.35,1.97,-3.38,-63.14
